# 과제 해설 (EXERCISE SOLUTION)

1일 차 프로젝트였던 '웹페이지 요약하기'를 업그레이드하여, OpenAI 대신 **Ollama를 통해 로컬에서 실행되는 오픈 소스 모델**을 사용하도록 수정하는 정답 예시입니다.

유료 API를 사용하지 않고 싶다면, 앞으로의 모든 프로젝트에 이 기술을 활용할 수 있습니다.

**장점:**
1. 비용 없음 - 오픈 소스 모델 활용
2. 데이터 보안 - 데이터가 내 컴퓨터 외부로 나가지 않음

**단점:**
1. 프론티어 모델(GPT-4 등)에 비해 현저히 낮은 성능

---

## Ollama 설치 방법 복습

[ollama.com](https://ollama.com)을 방문하여 설치하기만 하면 됩니다!

설치가 완료되면 Ollama 서버가 로컬에서 이미 실행 중이어야 합니다. 아래 주소에 접속하여 확인해 보세요:
[http://localhost:11434/](http://localhost:11434/)

`Ollama is running`이라는 메시지가 보이면 성공입니다.

만약 보이지 않는다면, 터미널(Mac)이나 Powershell(Windows)을 열고 `ollama serve`를 입력한 뒤 다시 접속해 보세요.

In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [2]:
# Constants

MODEL = "llama3.2"

In [3]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190 

## 프롬프트의 종류 (Types of prompts)

이미 알고 계실 수도 있지만, 모른다 하더라도 곧 매우 익숙해지실 내용입니다!

GPT-4o와 같은 모델들은 특정한 방식으로 지시를 받도록 훈련되었습니다.

모델은 기본적으로 다음 두 가지를 받기를 기대합니다:

1. **시스템 프롬프트 (System prompt):** 모델에게 어떤 **역할(Task)**을 수행해야 하는지, 그리고 어떤 **말투(Tone)**를 사용해야 하는지 알려주는 지침서입니다.
2. **사용자 프롬프트 (User prompt):** 실제 대화의 시작점이며, 모델이 **답변해야 할 구체적인 질문이나 요청**입니다.

In [15]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown. All answers will be in Korean."

In [16]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too. \
All answers will be in Korean.\n\n"
    user_prompt += website.text
    return user_prompt

## 메시지 구조 (Messages)

Ollama API는 OpenAI와 동일한 메시지 형식을 사용합니다:

```json
[
    {"role": "system", "content": "여기에 시스템 메시지를 입력합니다"},
    {"role": "user", "content": "여기에 사용자 메시지를 입력합니다"}
]

In [17]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## 하나로 합치기 - 이제 OpenAI 대신 Ollama를 사용합니다

In [18]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [19]:
summarize("https://edwarddonner.com")

'**Edward Donner의 웹사이트 요약**\n==========================\n\n*   **AI Curriculum**: LLMs(Large Language Models) 교육(curriculum) link.\n*   **Proficient AI Engineer**: AI 엔지니어 교육((course) link).\n*   **Connect Four**, **Outsmart**: AILiveEvent 및 n8n에關한文章.\n*   **About**: Edward Donner이란 사람의 소개(about), Neptune.io와 AI startups에關한 전기, Udemy 코스에關한 내용.\n*   **Posts**: articles나 blog post link. \n*   **News/Announcements**:\n    *   2026년 2월 17일: Vibe Coder -> Agentic Engineer\n    *   2026년 1월 4일: AI Builder(n8n) - Create Agent, Voice Agent\n    *   2025년 11월 11일: n8n을使用하여 Agent를 생성하고Voice Agent를 생성하도록합니다.\n    *   2025년 9월 15일: AI Live Event의 유니크 에너지\n    *   2025년 9월 15일: AI Engineering MLOps Track - Production에Deployment\n\n위 정보를 참고하세요.'

In [20]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [21]:
display_summary("https://edwarddonner.com")

**이bsites 요약**

* **제목**: 에드워드 Дон너 (Edward Donner)
* **내용**:
	+ AI 커리큘럼 (AI Curriculum) - LLMs와 관련된课程들
	+-proficient AI Engineer- 에드워드 Donovan의 프로필 page
	+ Connect Four - LLMs 간의 경쟁을 특징으로 하는 게임장소
	+ Outsmart - LLMs 간의 지략과 외모적 전략을 통해 승리하는 게임장소
	+ About - 에드워드 Donovan의 소개-page (코-CEO 인 Nebula.io, AI 스타트업 untapt)
* **신고/고고고**:
	+ 2021년 AI 스타트업 untapt가 acquired되었습니다. (2021)
	+ "LLMs에 대한 impromptu lecture"를 주제로의 Udemy 코스 - best-selling, top-rated 코스를 attainment
* **연결**:
	+ LinkedIn - 에드워드 Donovan의 LinkedIn 프로필
	+ Twitter - 에드워드 Donovan의 Twitter 계정
	+ Facebook - 에드워드 Donovan의 Facebookโปร필

# 다른 웹사이트들도 시도해 봅시다

주의: 이 방식은 아주 단순한 스크래핑 기법을 사용하므로, 모든 사이트에서 작동하지는 않습니다.

1. **JavaScript로 렌더링되는 사이트:** React 앱처럼 브라우저에서 실행되어야 내용이 보이는 사이트들은 이 코드로 내용을 가져올 수 없습니다. 이를 해결하려면 `community-contributions` 폴더에 있는 **Selenium** 구현 사례를 참고하세요. (설치 방법이 궁금하다면 저에게 물어봐 주세요!)
2. **보안 솔루션 적용 사이트:** CloudFront 등 보안 서비스로 보호되는 사이트는 `403 Forbidden` 에러를 보낼 수 있습니다. (이 점을 짚어주신 Andy J님께 감사드립니다!)

하지만 여전히 많은 웹사이트가 이 방식으로 아주 잘 작동합니다!

In [22]:
display_summary("https://cnn.com")

# ** cnn.com**

## ** Latest News and Articles**

*   **US Politics**: 
    *   Trump claims Obama revealed classified information on aliens.
    *   Former Prince Andrew leaves police station after arrest.
*   **Business**:
    *   JPMorgan Chase pushes back on Trump's $5 billion lawsuit, arguing it 'fraudulently' includes Jamie Dimon.
    *   Microsoft pledges $50 billion to tackle AI inequality as it warns of a 'growing divide'.
*   **Entertainment**:
    *   Eric Dane, ‘Grey’s Anatomy’ and ‘Euphoria’ star, dead at 53.
    *   Pheelz on AI: Why music still needs a soul.
*   **World News**:
    *   Satellite pictures show Iran is bracing for war.
    *   India opens 'world's biggest' AI summit, but what’s next?

## **Lifestyle**

*   **Health**: 
    *   Tracking US flu cases in maps and charts.
    *   A visual guide to the evidence in the Nancy Guthrie investigation
*   **Style**:
    *   King Charles makes a surprise visit to one of London’s biggest events.
    *   One Look With: Designer Tolu Coker backstage at London Fashion Week

## **Sports**

*   **Football**: 
    *   Can the US men's hockey team make the final?
    *   Big snowstorm could hit the East Coast this weekend
*   **Tennis**:
    *   Alysa Liu wins gold in Winter Olympics

## **Science**

*   **Space**: 
    *   NASA designates botched Boeing Starliner test flight a ‘Type A mishap’ in new report.
    *   Ancient bone may be first physical evidence of Hannibal’s 'war machine' elephants in Western Europe

## **Technology**:
*   Microsoft pledges $50 billion to tackle AI inequality as it warns of a 'growing divide'
*   Ad Feedback

In [23]:
display_summary("https://anthropic.com")

**안throp릭(Anthropic)의 website 요약**

안throp릭은 공공 유기체로 AI의ประโยชน과 위험을 보장하고 조절하는 목적으로 설립된 회사입니다. 안throp릭은 AI가 세계에 큰 영향을 미칠 것이라는 믿음을 가지고 있으며, 안throp릭의 사적 이익만을 목표로 삼지 않고Humanity의 장기적인 복지를 위해 AI를 개발하고 사용할 것을 목표로 합니다.

**최근 발표**

* 2026년 2월 4일: Claude Opus 4.6 model이 출시되었습니다. 이 modelo는 coding, agents 및 전문งาน에서 최고의 성능을 제공합니다.
* 2026년 2월 5일: Claude Sonnet 4.6 model이 출시되었습니다. 이 모델은 coding, agents 및 전문งาน에서 최적의 성능을 제공합니다.
* 2026년 2월 17일: 안throp릭의 responsibly scaling policy가 발표되었습니다.

**기대된 서비스**

* Claude Code
* Claude Developer Platform
* Claude in Chrome, Excel, PowerPoint, Slack
* Cowork
* Download app

**Products**

* Opus
* Sonnet
* Haiku

# 코드 공유하기

여러분이 작성한 코드를 나중에 공유해 주시면 다른 분들과 함께 나눌 수 있어 정말 기쁠 것 같습니다! 이미 `community-contributions` 폴더에는 다른 수강생분들이 개선한 내용(예: Selenium 구현 등)이 포함되어 있습니다. 여러분의 개선 버전을 이 폴더에 추가하고 싶다면, **Pull Request(PR)**를 보내주세요. 확인 후 제가 병합(Merge)하도록 하겠습니다.

만약 Git 사용이 익숙하지 않으시더라도 걱정 마세요(저도 전문가는 아닙니다!). AI 친구가 Pull Request를 보내는 법을 아주 친절하게 설명해 주었습니다. 과정이 조금 복잡해 보일 수 있지만, 한 번 해보고 나면 금방 익숙해질 것입니다.

**전문가의 팁(Pro-tip):** 노트북을 제출하기 전에는 주피터 노트북의 출력을 지워주시는 것이 좋습니다. (**Edit >> Clear all outputs** 메뉴를 선택한 후 **저장**) 이렇게 하면 훨씬 깔끔한 코드를 공유할 수 있습니다.

AI 친구가 알려주는 PR 방법 안내: [https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c](https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c)